# econchile — walkthrough

**Chilean macroeconomic data in a few lines.** This notebook shows *every* public function of the library, what it does, and a working example.

**How to run:** `pip install econchile` first, then run the cells top to bottom. The first ~10 cells need **no token** — only the final "live data" section needs `BCCH_TOKEN` (free at https://si3.bcentral.cl/Siete/en/Siete/API).

In [ ]:
# Install from PyPI (in Colab: uncomment the line below)
# !pip install -q econchile

import econchile
from econchile import BcchClient, Series
from econchile.offline import OfflineClient

print("econchile", econchile.__version__)

## 1. The catalog — `list_series()` and `search()`

The library ships 7 core series. `list_series()` shows them all; `search()` finds them by keyword (case- and accent-insensitive, matches Spanish/English titles too). **No token needed.**

In [ ]:
client = BcchClient()  # constructs fine without a token (v0.1.2+)

print("— ALL 7 SERIES —")
for meta in client.list_series():
    print(f"{meta.series_id:<30} {meta.spanish_title:<45} {meta.frequency.value}")

In [ ]:
print("— SEARCH —")
for meta in client.search("dolar"):
    print(meta.series_id, "|", meta.spanish_title)
print()
for meta in client.search("ipc"):
    print(meta.series_id, "|", meta.spanish_title)

## 2. The `Series` enum — human names for BCCh codes

`Series.UF` is a `str` enum: it *is* the BCCh code, and it carries metadata.

In [ ]:
print("Series.UF        =", Series.UF)
print("Series.UF.value  =", Series.UF.value, " <- the real BCCh code")
print("Series.USD.meta().english_title =", Series.USD.meta().english_title)
print("Series.from_code('F073.TCO.PRE.Z.D') =", Series.from_code("F073.TCO.PRE.Z.D"))
print("Series.list_all() =", [s.name for s in Series.list_all()])

## 3. THE basic call — one line of real work

`get(series, desde, hasta)` — that's it. Dates are `YYYY-MM-DD`. Returns a typed `SeriesResult`.

*No token yet? The next cell shows the clean error; set `BCCH_TOKEN` and re-run it for real data.*

In [ ]:
import os

if os.environ.get("BCCH_TOKEN"):
    result = client.get(Series.UF, "2026-08-10", "2026-08-14")
    print(f"UF, 2026-08-10..14: {len(result.observations)} observations, source={result.source}")
    for obs in result.observations[:3]:
        print(f"   {obs.date}  {obs.value}")
else:
    try:
        client.get(Series.UF, "2026-08-10", "2026-08-14")
    except Exception as exc:
        print(f"No token -> clean, package-specific error: {type(exc).__name__}")
        print("   Set BCCH_TOKEN in the environment and re-run this cell for live data.")

## 4. What's inside a `SeriesResult`

Everything you get back: the series, typed observations (immutable), when it was fetched, where the data came from (`api` / `cache`), and metadata.

In [ ]:
if os.environ.get("BCCH_TOKEN"):
    r = client.get(Series.USD, "2026-08-10", "2026-08-14")
    print("series    :", r.series)
    print("source    :", r.source)
    print("fetched_at:", r.fetched_at)
    print("metadata  :", r.metadata["descripEsp"])
    print("observations (first 3):")
    for o in r.observations[:3]:
        print(f"   {o.date}  {o.value}")
    print(f"... total {len(r.observations)} observations")
else:
    print("Set BCCH_TOKEN to inspect a real SeriesResult.")

## 5. Errors are intentional

Bad input fails fast with clear, catchable exceptions:

In [ ]:
# unknown series -> KeyError
try:
    client.get("NOT.A.SERIES", "2026-01-01", "2026-01-31")
except KeyError as exc:
    print("unknown series ->", type(exc).__name__)

# bad date -> ValueError
try:
    client.get(Series.UF, "01/2026", "2026-01-31")
except ValueError as exc:
    print("bad date      ->", type(exc).__name__, "|", exc)

## 6. `OfflineClient` — the resilience layer

Same idea as `BcchClient`, but API-first with a cache fallback: if the API fails (outage, rate limit), it serves the **last cached result** for that exact query — and only raises `BcchOfflineError` when both are exhausted. Perfect for cron jobs.

*(Note: `OfflineClient.get()` takes the same `series, desde, hasta` — it has no `use_cache` parameter; it always tries the API first.)*

In [ ]:
offline = OfflineClient()  # also constructs without a token

# No token + empty cache -> BcchOfflineError (clean, not a crash)
try:
    offline.get(Series.USD, "2026-08-10", "2026-08-14")
except Exception as exc:
    print("offline, empty cache ->", type(exc).__name__)
    print("   (after a successful fetch, this same call would serve the cache)")

In [ ]:
# OfflineClient cache fallback, demonstrated with a token:
# 1) fetch once with BcchClient  -> stores the result in the cache
# 2) ask OfflineClient for the SAME query -> API works, returns fresh data;
#    if the API were down, this exact call would serve the cached copy instead.
if os.environ.get("BCCH_TOKEN"):
    client.get(Series.UF, "2026-08-10", "2026-08-14")      # fills the cache
    r = offline.get(Series.UF, "2026-08-10", "2026-08-14")  # API-first, falls back to cache on failure
    print(f"offline.get() -> {len(r.observations)} obs, source={r.source}")
else:
    print("Set BCCH_TOKEN to see the offline fallback in action.")

## 7. The cache — repeat queries are instant

`BcchClient` is **cache-first**: within the TTL (24h by default), the same query is served from local SQLite (`~/.econchile/cache.db`) with zero network calls. You can change the TTL per client: `BcchClient(ttl_seconds=3600)` keeps entries for 1 hour.

In [ ]:
if os.environ.get("BCCH_TOKEN"):
    import time

    t0 = time.perf_counter()
    r1 = client.get(Series.UF, "2026-08-10", "2026-08-14", use_cache=False)  # force API
    t_api = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    r2 = client.get(Series.UF, "2026-08-10", "2026-08-14")  # cache hit
    t_cache = (time.perf_counter() - t0) * 1000

    print(f"first call  (API)   : {t_api:7.1f} ms")
    print(f"second call (cache): {t_cache:7.1f} ms")

    n = client.clear_cache()
    print(f"clear_cache() removed {n} row(s)")
else:
    print("Set BCCH_TOKEN to see the cache speedup demo.")

## 8. Sanity check vs the official source

Library values are **the official BCCh values** — verified against the published tables (e.g. dólar observado 17-08-2026 = $913,15).

In [ ]:
if os.environ.get("BCCH_TOKEN"):
    r = client.get(Series.USD, "2026-08-17", "2026-08-17", use_cache=False)
    obs = r.observations[0]
    print(f"USD 2026-08-17: library={obs.value}  official=913.15  match={obs.value == 913.15}")
else:
    print("Set BCCH_TOKEN to run the live sanity check.")